# LAB 3 — RAGAS + LangFuse Scores API: Monitoração Contínua de Qualidade

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Avaliação de Pipelines RAG com RAGAS e LangFuse**

---

## Objetivo

Integrar o cálculo **RAGAS** com envio automático ao **LangFuse Scores API** a cada execução do pipeline RAG — transformando observabilidade reativa em **monitoração contínua de qualidade**.

Enquanto o LAB 2 avaliou o dataset completo em lote, este laboratório instrumenta o pipeline para avaliar **cada consulta individualmente**, em tempo real, registrando métricas de qualidade diretamente nos traces do LangFuse.

## Conceito: Quality Gate em Produção

Um **quality gate** é um ponto de verificação automático que bloqueia ou alerta quando a qualidade cai abaixo de um limiar mínimo. Em sistemas RAG jurídicos, isso é crítico:

- Uma resposta com **Faithfulness < 0.80** pode conter informações não sustentadas pelo corpus — risco de induzir erro jurídico.
- **Context Recall < 0.70** indica que documentos importantes não foram recuperados — lacuna de cobertura.
- Monitorar **por trace** permite detectar degradação antes que o usuário perceba.

## Fluxo: RAG + Avaliação + Observabilidade

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     PIPELINE RAG COM AVALIAÇÃO                         │
│                                                                         │
│  Pergunta do                                                            │
│  Usuário       ──► [LangFuse Trace]                                     │
│                         │                                               │
│                         ├──► [Span: Retrieval]                          │
│                         │         │                                     │
│                         │         ▼                                     │
│                         │    OpenSearch kNN                             │
│                         │    BGE-M3 (dim=1024)                          │
│                         │         │                                     │
│                         │    k=5 chunks ──────────────────────────────►│
│                         │                                              contexts
│                         ├──► [Span: Geração]                            │
│                         │         │                                     │
│                         │         ▼                                     │
│                         │    vLLM Llama 3.1 8B                          │
│                         │    (localhost:8000/v1)                        │
│                         │         │                                     │
│                         │    answer ───────────────────────────────────►│
│                         │                                              answer
│                         ├──► [RAGAS evaluate()]                         │
│                         │         │                                     │
│                         │    faithfulness          ────► LangFuse Score │
│                         │    answer_relevancy       ────► LangFuse Score│
│                         │    context_recall         ────► LangFuse Score│
│                         │    context_precision      ────► LangFuse Score│
│                         │                                               │
│                         └──► trace_id visível na UI                     │
└─────────────────────────────────────────────────────────────────────────┘
```

## Pré-requisitos

| Componente | Endereço | Observação |
|---|---|---|
| vLLM (Llama 3.1 8B Instruct) | `localhost:8000/v1` | LLM gerador e judge RAGAS |
| OpenSearch 3.x | `localhost:9200` | Índice `juridico_baseline_aula4` |
| LangFuse | `localhost:3000` | Scores API e UI de observabilidade |
| Dataset do LAB 1 | `dataset_avaliacao_completo.json` | 50 pares com ground truth |
| Embedding | `BAAI/bge-m3` (dim=1024) | Vetorização kNN |

## Referência Normativa

Conforme **ABNT NBR 6023:2018**.  
ES-JOLLY, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.  
LANGFUSE. **Scores API Documentation**. 2024. Disponível em: <https://langfuse.com/docs/scores>. Acesso em: abr. 2026.  
Meta. **Llama 3.1**. 2024. Disponível em: <https://llama.meta.com>. Acesso em: abr. 2026.

---
## Célula 1 — Instalação de Dependências

In [ ]:
# Instalação das dependências necessárias para o LAB 3
# Compatível com Python 3.11+ e Google Colab T4
!pip install -q ragas langfuse langchain langchain-openai sentence-transformers opensearch-py datasets pandas matplotlib

# Verifica versões instaladas
import importlib
pacotes = ["ragas", "langfuse", "langchain", "langchain_openai",
           "sentence_transformers", "opensearchpy", "datasets", "pandas", "matplotlib"]

print("Dependências instaladas:")
for pacote in pacotes:
    try:
        mod = importlib.import_module(pacote)
        versao = getattr(mod, "__version__", "N/A")
        print(f"  {pacote:<25} v{versao}")
    except ImportError:
        print(f"  {pacote:<25} NÃO INSTALADO")

print("\nInstalação concluída.")

---
## Célula 2 — Configuração de Variáveis de Ambiente

In [ ]:
import os

# ── OpenSearch ─────────────────────────────────────────────────────────────
os.environ["OPENSEARCH_HOST"] = os.getenv("OPENSEARCH_HOST", "localhost")
os.environ["OPENSEARCH_PORT"] = os.getenv("OPENSEARCH_PORT", "9200")
os.environ["OPENSEARCH_USER"] = os.getenv("OPENSEARCH_USER", "admin")
os.environ["OPENSEARCH_PASS"] = os.getenv("OPENSEARCH_PASS", "admin")
os.environ["OPENSEARCH_INDEX"] = os.getenv("OPENSEARCH_INDEX", "juridico_baseline_aula4")

# ── vLLM ───────────────────────────────────────────────────────────────────
os.environ["VLLM_BASE_URL"] = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
os.environ["VLLM_MODEL"]    = os.getenv("VLLM_MODEL",    "meta-llama/Meta-Llama-3.1-8B-Instruct")

# ── LangFuse ───────────────────────────────────────────────────────────────
os.environ["LANGFUSE_HOST"]       = os.getenv("LANGFUSE_HOST",       "http://localhost:3000")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-SUBSTITUA")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-SUBSTITUA")

# ── Embedding ──────────────────────────────────────────────────────────────
os.environ["EMBEDDING_MODEL"] = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Constantes de uso local ─────────────────────────────────────────────────
VLLM_BASE_URL    = os.environ["VLLM_BASE_URL"]
VLLM_MODEL       = os.environ["VLLM_MODEL"]
EMBEDDING_MODEL  = os.environ["EMBEDDING_MODEL"]
LANGFUSE_HOST    = os.environ["LANGFUSE_HOST"]
OPENSEARCH_INDEX = os.environ["OPENSEARCH_INDEX"]
DATASET_LAB1     = "dataset_avaliacao_completo.json"

# Metas mínimas do syllabus (quality gate)
METAS = {
    "faithfulness":      0.80,
    "answer_relevancy":  0.75,
    "context_recall":    0.70,
    "context_precision": 0.70,
}

print("Variáveis de ambiente configuradas.")
print(f"  vLLM       : {VLLM_BASE_URL}")
print(f"  Modelo     : {VLLM_MODEL}")
print(f"  Embedding  : {EMBEDDING_MODEL}")
print(f"  LangFuse   : {LANGFUSE_HOST}")
print(f"  OpenSearch : {os.environ['OPENSEARCH_HOST']}:{os.environ['OPENSEARCH_PORT']}")
print(f"  Índice     : {OPENSEARCH_INDEX}")
print(f"\nMetas mínimas (quality gate):")
for metrica, meta in METAS.items():
    print(f"  {metrica:<22} : >= {meta}")

---
## Célula 3 — Conexões e Health Checks

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from opensearchpy import OpenSearch
from langchain_openai import ChatOpenAI
from langfuse import Langfuse
from sentence_transformers import SentenceTransformer

# ── OpenSearch ─────────────────────────────────────────────────────────────
OPENSEARCH_OK = False
os_client = None
try:
    os_client = OpenSearch(
        hosts=[{"host": os.environ["OPENSEARCH_HOST"],
                "port": int(os.environ["OPENSEARCH_PORT"])}],
        http_auth=(os.environ["OPENSEARCH_USER"], os.environ["OPENSEARCH_PASS"]),
        use_ssl=False,
        verify_certs=False,
        timeout=30,
    )
    info = os_client.info()
    OPENSEARCH_OK = True
    print(f"[OK] OpenSearch conectado — versão: {info['version']['number']}")

    # Verifica índice
    if os_client.indices.exists(index=OPENSEARCH_INDEX):
        stats = os_client.indices.stats(index=OPENSEARCH_INDEX)
        total_docs = stats["_all"]["primaries"]["docs"]["count"]
        print(f"  Índice '{OPENSEARCH_INDEX}': {total_docs} documentos")
    else:
        print(f"  [AVISO] Índice '{OPENSEARCH_INDEX}' não encontrado.")
        print("  Execute o LAB 1 para criar e popular o índice.")

except Exception as exc:
    print(f"[AVISO] OpenSearch não disponível: {exc}")

# ── vLLM ───────────────────────────────────────────────────────────────────
VLLM_OK = False
llm = None
try:
    llm = ChatOpenAI(
        model=VLLM_MODEL,
        base_url=VLLM_BASE_URL,
        api_key="dummy",
        temperature=0.1,
        max_tokens=512,
    )
    resp = llm.invoke("Responda apenas: OK")
    VLLM_OK = True
    print(f"[OK] vLLM conectado — resposta: {resp.content.strip()[:40]}")
except Exception as exc:
    print(f"[AVISO] vLLM não disponível: {exc}")

# ── LangFuse ───────────────────────────────────────────────────────────────
LANGFUSE_OK = False
lf_client = None
try:
    lf_client = Langfuse(
        public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
        secret_key=os.environ["LANGFUSE_SECRET_KEY"],
        host=LANGFUSE_HOST,
    )
    lf_client.auth_check()
    LANGFUSE_OK = True
    print(f"[OK] LangFuse conectado em {LANGFUSE_HOST}")
except Exception as exc:
    print(f"[AVISO] LangFuse não disponível: {exc}")

# ── Embedder BGE-M3 ────────────────────────────────────────────────────────
EMBEDDER_OK = False
embedder = None
try:
    print(f"Carregando embedder '{EMBEDDING_MODEL}'...")
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    # Teste de dimensionalidade
    vetor_teste = embedder.encode("teste", normalize_embeddings=True)
    assert len(vetor_teste) == 1024, f"Dimensão inesperada: {len(vetor_teste)}"
    EMBEDDER_OK = True
    print(f"[OK] Embedder carregado — dim: {len(vetor_teste)}")
except Exception as exc:
    print(f"[AVISO] Embedder não disponível: {exc}")

# ── Sumário ────────────────────────────────────────────────────────────────
print("\nSaúde dos serviços:")
servicos = {
    "OpenSearch": OPENSEARCH_OK,
    "vLLM"      : VLLM_OK,
    "LangFuse"  : LANGFUSE_OK,
    "Embedder"  : EMBEDDER_OK,
}
for nome, ok in servicos.items():
    status = "OK" if ok else "INDISPONIVEL (modo degradado)"
    print(f"  {nome:<12}: {status}")

---
## Célula 4 — Classe `RAGPipelineComAvaliacao`

Esta classe encapsula todo o fluxo: recuperação → geração → avaliação RAGAS → envio ao LangFuse Scores API.  
Cada chamada a `run()` cria um trace completo no LangFuse com spans de retrieval e geração, além de registrar as 4 métricas RAGAS como scores vinculados ao trace.

In [ ]:
import json
import time
import datetime
import math
from typing import Optional

import pandas as pd
from datasets import Dataset
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_core.embeddings import Embeddings


# ── Wrapper de Embeddings para RAGAS (usando SentenceTransformer local) ────
class STEmbeddingsWrapper(Embeddings):
    """Wrapper LangChain para SentenceTransformer, compatível com RAGAS."""

    def __init__(self, model_name: str):
        self._model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list) -> list:
        return self._model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text: str) -> list:
        return self._model.encode(text, normalize_embeddings=True).tolist()


class RAGPipelineComAvaliacao:
    """
    Pipeline RAG com avaliação RAGAS integrada e envio automático
    ao LangFuse Scores API.

    Fluxo por consulta:
      1. Cria trace no LangFuse
      2. Span de retrieval: busca kNN com BGE-M3 no OpenSearch
      3. Span de geração: gera resposta com vLLM
      4. Avalia com RAGAS (4 métricas)
      5. Envia cada score ao LangFuse Scores API
    """

    def __init__(
        self,
        os_client,
        llm: ChatOpenAI,
        embedder: SentenceTransformer,
        langfuse: Optional[object],
        indice: str = "juridico_baseline_aula4",
        k: int = 5,
    ):
        """
        Parâmetros:
            os_client : cliente OpenSearch
            llm       : modelo de linguagem (vLLM via LangChain)
            embedder  : SentenceTransformer BGE-M3
            langfuse  : cliente LangFuse (None se indisponível)
            indice    : nome do índice OpenSearch
            k         : número de chunks recuperados
        """
        self.os_client = os_client
        self.llm       = llm
        self.embedder  = embedder
        self.lf        = langfuse
        self.indice    = indice
        self.k         = k

        # Configura LLM e embeddings para o RAGAS
        ragas_llm = LangchainLLMWrapper(
            ChatOpenAI(
                model=VLLM_MODEL,
                base_url=VLLM_BASE_URL,
                api_key="dummy",
                temperature=0.0,  # determinístico para avaliação
                max_tokens=1024,
            )
        )
        ragas_emb = LangchainEmbeddingsWrapper(STEmbeddingsWrapper(EMBEDDING_MODEL))

        # Configura cada métrica RAGAS com LLM e embeddings
        self.metricas = [
            faithfulness,
            answer_relevancy,
            context_recall,
            context_precision,
        ]
        for metrica in self.metricas:
            metrica.llm = ragas_llm
            if hasattr(metrica, "embeddings"):
                metrica.embeddings = ragas_emb

    # ── Retrieval ──────────────────────────────────────────────────────────
    def _recuperar_contextos(self, question: str) -> list:
        """
        Busca kNN no OpenSearch usando embedding BGE-M3.
        Retorna lista de strings com o conteúdo dos chunks recuperados.
        """
        if self.os_client is None or self.embedder is None:
            # Fallback quando serviços não estão disponíveis
            return [
                f"Contexto de fallback para: {question[:80]}. "
                "Execute com OpenSearch e embedder para resultados reais."
            ]

        # Gera embedding da query com BGE-M3
        vetor_query = self.embedder.encode(
            question, normalize_embeddings=True
        ).tolist()

        # Consulta kNN no OpenSearch
        query_knn = {
            "size": self.k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": vetor_query,
                        "k": self.k,
                    }
                }
            },
            "_source": ["conteudo", "titulo", "artigo", "tipo"],
        }

        try:
            resposta = self.os_client.search(
                index=self.indice,
                body=query_knn,
            )
            contextos = [
                hit["_source"].get("conteudo", "").strip()
                for hit in resposta["hits"]["hits"]
                if hit["_source"].get("conteudo", "").strip()
            ]
            return contextos if contextos else ["Nenhum contexto relevante encontrado."]
        except Exception as exc:
            return [f"Erro na busca kNN: {exc}"]

    # ── Geração ────────────────────────────────────────────────────────────
    def _gerar_resposta(self, question: str, contextos: list) -> str:
        """
        Gera resposta com vLLM usando os contextos recuperados.
        """
        if self.llm is None:
            return (
                "Resposta de fallback: vLLM não disponível. "
                "Configure o servidor vLLM em localhost:8000."
            )

        contexto_texto = "\n\n".join(
            f"[Documento {i+1}]\n{ctx}" for i, ctx in enumerate(contextos)
        )

        prompt = (
            "Você é um assistente jurídico especializado em Direito Brasileiro e "
            "Segurança Pública. Responda à pergunta abaixo com base EXCLUSIVAMENTE "
            "nos documentos fornecidos. Seja preciso, cite o dispositivo legal "
            "pertinente e evite afirmações não sustentadas pelos textos.\n\n"
            f"DOCUMENTOS:\n{contexto_texto}\n\n"
            f"PERGUNTA: {question}\n\n"
            "RESPOSTA:"
        )

        try:
            resp = self.llm.invoke(prompt)
            return resp.content.strip()
        except Exception as exc:
            return f"Erro na geração: {exc}"

    # ── Avaliação RAGAS ────────────────────────────────────────────────────
    def _avaliar_ragas(self, question: str, answer: str,
                       contextos: list, ground_truth: str) -> dict:
        """
        Calcula as 4 métricas RAGAS para um par (question, answer, contexts, ground_truth).
        Retorna dicionário com os scores ou valores NaN em caso de erro.
        """
        scores = {m.name: float("nan") for m in self.metricas}

        # Cria o dataset de 1 par para avaliação
        dataset_par = Dataset.from_dict({
            "question":     [question],
            "answer":       [answer],
            "contexts":     [contextos],
            "ground_truth": [ground_truth],
        })

        try:
            resultado = evaluate(
                dataset=dataset_par,
                metrics=self.metricas,
                raise_exceptions=False,
            )
            for nome in scores:
                try:
                    val = resultado[nome]
                    scores[nome] = float(val) if not math.isnan(float(val)) else 0.0
                except Exception:
                    scores[nome] = 0.0
        except Exception as exc:
            print(f"    [AVISO RAGAS] {exc}")
            # Tenta calcular métricas individualmente
            for metrica in self.metricas:
                try:
                    res_ind = evaluate(
                        dataset=dataset_par,
                        metrics=[metrica],
                        raise_exceptions=False,
                    )
                    scores[metrica.name] = float(res_ind[metrica.name])
                except Exception:
                    scores[metrica.name] = 0.0

        return scores

    # ── run() — execução completa de 1 par ────────────────────────────────
    def run(self, question: str, ground_truth: str) -> dict:
        """
        Executa o pipeline completo para uma pergunta:
          (a) Cria trace no LangFuse
          (b) Span de retrieval com kNN BGE-M3
          (c) Span de geração com vLLM
          (d) Calcula RAGAS nas 4 métricas
          (e) Envia cada métrica via langfuse.score()
          (f) Retorna dict com answer, contexts, scores, trace_id

        Parâmetros:
            question     : pergunta jurídica
            ground_truth : resposta esperada (ground truth do dataset)

        Retorno:
            dict com chaves: answer, contexts, scores, trace_id
        """
        ts_inicio = time.time()

        # ── (a) Cria trace no LangFuse ─────────────────────────────────────
        trace_id = None
        trace    = None
        if self.lf:
            try:
                trace = self.lf.trace(
                    name="RAG-Pipeline-Juridico",
                    input={"question": question, "ground_truth": ground_truth[:100]},
                    metadata={
                        "indice":    self.indice,
                        "k":         self.k,
                        "modelo":    VLLM_MODEL,
                        "embedding": EMBEDDING_MODEL,
                        "lab":       "LAB3",
                    },
                    tags=["rag", "juridico", "lab3", "ragas-integrado"],
                )
                trace_id = trace.id
            except Exception as exc:
                print(f"  [AVISO] Erro ao criar trace: {exc}")

        # ── (b) Span de retrieval ──────────────────────────────────────────
        ts_ret_inicio = time.time()
        contextos = self._recuperar_contextos(question)
        ts_ret_fim = time.time()

        if trace:
            try:
                trace.span(
                    name="retrieval-knn-bge-m3",
                    input={"question": question, "k": self.k},
                    output={"n_contextos": len(contextos),
                            "preview": contextos[0][:150] if contextos else ""},
                    metadata={
                        "latencia_ms": round((ts_ret_fim - ts_ret_inicio) * 1000, 2),
                        "indice":      self.indice,
                        "embedding":   EMBEDDING_MODEL,
                        "dim":         1024,
                    },
                    start_time=datetime.datetime.fromtimestamp(ts_ret_inicio),
                    end_time=datetime.datetime.fromtimestamp(ts_ret_fim),
                )
            except Exception as exc:
                print(f"  [AVISO] Erro ao criar span retrieval: {exc}")

        # ── (c) Span de geração ────────────────────────────────────────────
        ts_gen_inicio = time.time()
        answer = self._gerar_resposta(question, contextos)
        ts_gen_fim = time.time()

        if trace:
            try:
                trace.span(
                    name="geracao-vllm-llama",
                    input={"question": question, "n_contextos": len(contextos)},
                    output={"answer": answer[:300]},
                    metadata={
                        "latencia_ms": round((ts_gen_fim - ts_gen_inicio) * 1000, 2),
                        "modelo":      VLLM_MODEL,
                        "temperatura": 0.1,
                        "max_tokens":  512,
                    },
                    start_time=datetime.datetime.fromtimestamp(ts_gen_inicio),
                    end_time=datetime.datetime.fromtimestamp(ts_gen_fim),
                )
            except Exception as exc:
                print(f"  [AVISO] Erro ao criar span geração: {exc}")

        # ── (d) Avaliação RAGAS ────────────────────────────────────────────
        scores = self._avaliar_ragas(question, answer, contextos, ground_truth)

        # ── (e) Envio ao LangFuse Scores API ──────────────────────────────
        if self.lf and trace_id:
            for nome_metrica, valor in scores.items():
                try:
                    meta_val = METAS.get(nome_metrica, 0.0)
                    status   = "ATINGIDA" if valor >= meta_val else "ABAIXO"
                    self.lf.score(
                        trace_id=trace_id,
                        name=nome_metrica,
                        value=valor,
                        comment=(
                            f"Meta: {meta_val} | Status: {status} | "
                            f"Diferença: {valor - meta_val:+.4f}"
                        ),
                    )
                except Exception as exc:
                    print(f"  [AVISO] Erro ao enviar score '{nome_metrica}': {exc}")

            # Atualiza output do trace com resultado final
            try:
                trace.update(
                    output={
                        "answer":      answer[:300],
                        "scores":      scores,
                        "tempo_total": round(time.time() - ts_inicio, 2),
                    }
                )
                self.lf.flush()
            except Exception:
                pass

        # ── (f) Retorno ────────────────────────────────────────────────────
        return {
            "question":     question,
            "answer":       answer,
            "contexts":     contextos,
            "ground_truth": ground_truth,
            "scores":       scores,
            "trace_id":     trace_id,
            "tempo_s":      round(time.time() - ts_inicio, 2),
        }

    # ── run_batch() — processamento em lote ───────────────────────────────
    def run_batch(self, pares: list, max_pares: int = 10) -> list:
        """
        Processa múltiplos pares (question, ground_truth) sequencialmente.

        Parâmetros:
            pares     : lista de dicts com chaves 'question' e 'ground_truth'
            max_pares : limite de pares a processar (padrão: 10)

        Retorno:
            lista de resultados (um dict por par processado)
        """
        pares_proc = pares[:max_pares]
        resultados = []

        print(f"Processando {len(pares_proc)} pares (de {len(pares)} disponíveis)...")
        print(f"{'Par':<5} {'Faithfulness':>13} {'Ans.Rel':>10} {'Ctx.Rec':>10} {'Ctx.Pre':>10} {'Trace ID':<36} {'Tempo(s)':>9}")
        print("-" * 100)

        for i, par in enumerate(pares_proc, start=1):
            resultado = self.run(
                question=par["question"],
                ground_truth=par["ground_truth"],
            )
            resultados.append(resultado)

            s = resultado["scores"]
            tid = resultado["trace_id"] or "N/A"
            print(
                f"{i:<5} "
                f"{s.get('faithfulness', float('nan')):>13.4f} "
                f"{s.get('answer_relevancy', float('nan')):>10.4f} "
                f"{s.get('context_recall', float('nan')):>10.4f} "
                f"{s.get('context_precision', float('nan')):>10.4f} "
                f"{str(tid):<36} "
                f"{resultado['tempo_s']:>9.1f}s"
            )

        print("-" * 100)
        print(f"Batch concluído: {len(resultados)} pares processados.")
        return resultados


print("[OK] Classe RAGPipelineComAvaliacao definida com sucesso.")
print("  Métodos disponíveis:")
print("    run(question, ground_truth)       -> dict")
print("    run_batch(pares, max_pares=10)    -> list[dict]")

---
## Célula 5 — Instanciação do Pipeline e Carregamento do Dataset

In [ ]:
import json

# ── Carrega dataset do LAB 1 ───────────────────────────────────────────────
dataset_pares: list = []

try:
    with open(DATASET_LAB1, "r", encoding="utf-8") as f:
        dataset_pares = json.load(f)
    print(f"[OK] Dataset carregado: {len(dataset_pares)} pares")
    print(f"  Campos: {list(dataset_pares[0].keys())}")

except FileNotFoundError:
    print(f"[AVISO] '{DATASET_LAB1}' não encontrado. Gerando 15 pares sintéticos.")

    # Pares sintéticos representativos do domínio jurídico
    pares_sinteticos = [
        {
            "id": "q001", "tipo": "processual_penal",
            "question": "Qual o prazo para o oferecimento da denúncia pelo Ministério Público no processo penal ordinário?",
            "ground_truth": "Conforme o art. 46 do Código de Processo Penal, o Ministério Público tem 5 dias para oferecer denúncia quando o indiciado estiver preso, e 15 dias quando estiver solto.",
        },
        {
            "id": "q002", "tipo": "ambiental",
            "question": "Quais são as penalidades previstas na Lei de Crimes Ambientais para o crime de poluição?",
            "ground_truth": "A Lei 9.605/1998, art. 54, prevê reclusão de 1 a 4 anos para quem causar poluição de qualquer natureza capaz de causar danos à saúde humana, mortandade de animais ou destruição significativa da flora.",
        },
        {
            "id": "q003", "tipo": "constitucional",
            "question": "O que a Constituição Federal estabelece sobre a inviolabilidade do domicílio?",
            "ground_truth": "O art. 5º, XI, da CF/88 estabelece que a casa é asilo inviolável do indivíduo, ninguém nela podendo penetrar sem consentimento do morador, salvo em caso de flagrante delito, desastre, ou para prestar socorro, ou durante o dia, por determinação judicial.",
        },
        {
            "id": "q004", "tipo": "administrativo",
            "question": "Quais são os princípios da administração pública previstos na Constituição Federal?",
            "ground_truth": "O art. 37 da CF/88 estabelece os princípios da legalidade, impessoalidade, moralidade, publicidade e eficiência para a administração pública direta e indireta.",
        },
        {
            "id": "q005", "tipo": "processual_penal",
            "question": "Qual o procedimento para a prisão em flagrante?",
            "ground_truth": "O art. 306 do CPP determina que a prisão de qualquer pessoa e o local onde se encontre serão comunicados imediatamente ao juiz competente, ao Ministério Público e à família do preso, ou à pessoa por ele indicada.",
        },
        {
            "id": "q006", "tipo": "seguranca_publica",
            "question": "Quais são os órgãos de segurança pública previstos na Constituição Federal?",
            "ground_truth": "O art. 144 da CF/88 define como órgãos de segurança pública: a Polícia Federal, a Polícia Rodoviária Federal, a Polícia Ferroviária Federal, as Polícias Civis, as Polícias Militares, os Corpos de Bombeiros Militares e as Guardas Municipais.",
        },
        {
            "id": "q007", "tipo": "penal",
            "question": "Qual a pena prevista para o crime de homicídio doloso simples?",
            "ground_truth": "O art. 121 do Código Penal prevê pena de reclusão de 6 a 20 anos para o homicídio doloso simples. Se qualificado, a pena é de 12 a 30 anos de reclusão.",
        },
        {
            "id": "q008", "tipo": "ambiental",
            "question": "O que é o licenciamento ambiental e quando é obrigatório?",
            "ground_truth": "O licenciamento ambiental, conforme a Resolução CONAMA 237/1997 e a Lei Complementar 140/2011, é obrigatório para atividades ou empreendimentos potencialmente causadores de degradação ambiental, devendo ser obtido junto ao órgão ambiental competente.",
        },
        {
            "id": "q009", "tipo": "constitucional",
            "question": "Qual o prazo para impetrar mandado de segurança?",
            "ground_truth": "A Lei 12.016/2009, art. 23, estabelece prazo decadencial de 120 dias para impetrar mandado de segurança, contados da ciência do ato impugnado pelo interessado.",
        },
        {
            "id": "q010", "tipo": "processual_penal",
            "question": "O que é habeas corpus e quando pode ser utilizado?",
            "ground_truth": "O habeas corpus é remédio constitucional (art. 5º, LXVIII, CF/88) que visa proteger a liberdade de locomoção, podendo ser impetrado quando alguém sofrer ou se achar ameaçado de sofrer violência ou coação em sua liberdade de locomoção por ilegalidade ou abuso de poder.",
        },
        {
            "id": "q011", "tipo": "civil",
            "question": "Qual o prazo prescricional geral para ações de reparação civil?",
            "ground_truth": "O Código Civil, art. 206, §3º, V, estabelece prazo prescricional de 3 anos para a pretensão de reparação civil. O prazo geral, conforme art. 205, é de 10 anos.",
        },
        {
            "id": "q012", "tipo": "seguranca_publica",
            "question": "O que é o uso progressivo da força pelos policiais?",
            "ground_truth": "O uso progressivo da força é o princípio que orienta a atuação policial, pelo qual o agente deve empregar o nível de força estritamente necessário para controlar a situação, partindo de técnicas menos lesivas até as mais letais, conforme a Portaria Interministerial MJSP/MDH 4226/2010.",
        },
        {
            "id": "q013", "tipo": "administrativo",
            "question": "O que é o processo administrativo disciplinar e quais suas fases?",
            "ground_truth": "O PAD, regido pela Lei 8.112/1990, é o instrumento para apuração de irregularidades de servidores públicos federais, compreendendo as fases de instauração, inquérito administrativo (instrução, defesa e relatório) e julgamento.",
        },
        {
            "id": "q014", "tipo": "penal",
            "question": "Quais são as causas de exclusão da ilicitude no Código Penal?",
            "ground_truth": "O art. 23 do Código Penal prevê como excludentes de ilicitude: o estado de necessidade, a legítima defesa, o estrito cumprimento de dever legal e o exercício regular de direito.",
        },
        {
            "id": "q015", "tipo": "constitucional",
            "question": "Quais são os direitos fundamentais dos presos previstos na Constituição Federal?",
            "ground_truth": "O art. 5º, XLIX, da CF/88 garante aos presos o respeito à integridade física e moral. Outros direitos incluem: não ser submetido a tortura (III), contraditório e ampla defesa (LV), identificação dos responsáveis pela prisão (LXIV), assistência jurídica gratuita (LXXIV) e comunicação da prisão ao juiz e à família (LXII).",
        },
    ]
    dataset_pares = pares_sinteticos
    print(f"[OK] {len(dataset_pares)} pares sintéticos criados para demonstração.")

# ── Instancia o pipeline ───────────────────────────────────────────────────
pipeline = RAGPipelineComAvaliacao(
    os_client=os_client if OPENSEARCH_OK else None,
    llm=llm if VLLM_OK else None,
    embedder=embedder if EMBEDDER_OK else None,
    langfuse=lf_client if LANGFUSE_OK else None,
    indice=OPENSEARCH_INDEX,
    k=5,
)

print(f"\n[OK] Pipeline instanciado com {len(dataset_pares)} pares disponíveis.")
print(f"  Primeiras 3 perguntas:")
for i, par in enumerate(dataset_pares[:3], start=1):
    print(f"  {i}. [{par.get('tipo', 'N/A')}] {par['question'][:80]}...")

---
## Célula 6 — Demonstração com 10 Pares do Dataset

Processa 10 pares do dataset, exibindo o `trace_id` de cada execução para rastreabilidade no LangFuse.

In [ ]:
# Executa o pipeline em lote com 10 pares
# Cada execução cria um trace no LangFuse e calcula as 4 métricas RAGAS
resultados_batch = pipeline.run_batch(
    pares=dataset_pares,
    max_pares=10,
)

print(f"\nExemplo do resultado do Par 1:")
r = resultados_batch[0]
print(f"  Pergunta   : {r['question'][:80]}...")
print(f"  Resposta   : {r['answer'][:120]}...")
print(f"  Contextos  : {len(r['contexts'])} chunks recuperados")
print(f"  Trace ID   : {r['trace_id']}")
print(f"  Tempo      : {r['tempo_s']}s")
print(f"  Scores RAGAS:")
for metrica, valor in r['scores'].items():
    meta = METAS.get(metrica, 0.0)
    ok   = "OK" if valor >= meta else "ABAIXO"
    print(f"    {metrica:<22}: {valor:.4f} (meta: {meta}) [{ok}]")

---
## Célula 7 — Verificação no LangFuse: Traces e Scores

Como confirmar que os scores foram registrados corretamente na UI do LangFuse.

In [ ]:
# Exibe os links diretos para cada trace no LangFuse
# Acesse para verificar os scores nas 4 métricas RAGAS

print("="*70)
print("TRACES REGISTRADOS NO LANGFUSE")
print("="*70)
print(f"Host LangFuse: {LANGFUSE_HOST}")
print()

traces_com_id = [(i, r) for i, r in enumerate(resultados_batch, start=1) if r["trace_id"]]

if traces_com_id:
    for i, resultado in traces_com_id:
        trace_id = resultado["trace_id"]
        url = f"{LANGFUSE_HOST}/traces/{trace_id}"
        scores = resultado["scores"]
        print(f"Par {i:02d}: {url}")
        print(f"  Pergunta  : {resultado['question'][:70]}...")
        print(f"  Scores    : faithfulness={scores.get('faithfulness', 'N/A'):.4f}, "
              f"ans_relevancy={scores.get('answer_relevancy', 'N/A'):.4f}")
        print()
else:
    print("[AVISO] Nenhum trace registrado — LangFuse não está disponível.")
    print("  Os scores foram calculados localmente:")
    for i, resultado in enumerate(resultados_batch, start=1):
        print(f"  Par {i:02d}: {resultado['scores']}")

print()
print("Como confirmar scores na UI LangFuse:")
print("  1. Acesse o link do trace acima")
print("  2. Na aba 'Scores', verifique as 4 métricas RAGAS")
print("  3. Clique em 'Spans' para ver detalhes do retrieval e geração")
print("  4. Na aba 'Sessions', veja o histórico de todos os traces do LAB3")
print(f"  5. Dashboard geral: {LANGFUSE_HOST}/dashboard")

---
## Célula 8 — Dashboard de Qualidade: Médias vs Metas

Calcula as médias das 4 métricas sobre os 10 pares processados e gera um gráfico de barras comparando os valores obtidos com as metas mínimas.

In [ ]:
import statistics
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

# ── Calcula médias das métricas sobre os 10 pares ─────────────────────────
medias: dict = {}
desvios: dict = {}

for nome_metrica in METAS:
    valores = [
        r["scores"].get(nome_metrica, float("nan"))
        for r in resultados_batch
        if not math.isnan(r["scores"].get(nome_metrica, float("nan")))
    ]
    if valores:
        medias[nome_metrica]  = statistics.mean(valores)
        desvios[nome_metrica] = statistics.stdev(valores) if len(valores) > 1 else 0.0
    else:
        medias[nome_metrica]  = 0.0
        desvios[nome_metrica] = 0.0

# ── Tabela de resultados ───────────────────────────────────────────────────
print("=" * 75)
print("DASHBOARD DE QUALIDADE — 10 PARES — LAB 3")
print("=" * 75)
print(f"{'Métrica':<25} {'Média':>8} {'Desvio':>8} {'Meta':>8} {'Status':>10} {'Δ Meta':>8}")
print("-" * 75)

for nome, media in medias.items():
    meta    = METAS[nome]
    desvio  = desvios[nome]
    status  = "[OK]" if media >= meta else "[ABAIXO]"
    delta   = media - meta
    print(f"{nome:<25} {media:>8.4f} {desvio:>8.4f} {meta:>8.2f} {status:>10} {delta:>+8.4f}")

print("=" * 75)

n_atingidas = sum(1 for n, m in medias.items() if m >= METAS[n])
print(f"Metas atingidas: {n_atingidas}/4")
print(f"Pares avaliados: {len(resultados_batch)}")

# ── Gráfico de barras ──────────────────────────────────────────────────────
nomes_exib  = ["Faithfulness", "Ans. Relevancy", "Ctx. Recall", "Ctx. Precision"]
valores_bar = list(medias.values())
metas_bar   = list(METAS.values())
erros_bar   = list(desvios.values())
x           = range(len(nomes_exib))

# Cores: verde se atingiu meta, vermelho se abaixo
cores = [
    "#2ecc71" if v >= m else "#e74c3c"
    for v, m in zip(valores_bar, metas_bar)
]

fig, ax = plt.subplots(figsize=(10, 5))

barras = ax.bar(
    x, valores_bar, color=cores, alpha=0.85, width=0.5,
    yerr=erros_bar, capsize=5, error_kw={"ecolor": "#555", "lw": 1.5},
)

# Linha de meta (vermelha tracejada)
for i, (meta, nome) in enumerate(zip(metas_bar, nomes_exib)):
    ax.hlines(
        meta, i - 0.35, i + 0.35,
        colors="red", linestyles="--", linewidths=2,
    )

# Rótulos das barras
for barra, val in zip(barras, valores_bar):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.015,
        f"{val:.3f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

# Estética
ax.set_ylim(0, 1.15)
ax.set_xticks(list(x))
ax.set_xticklabels(nomes_exib, fontsize=11)
ax.set_ylabel("Score Médio (0–1)", fontsize=11)
ax.set_title(
    "Dashboard de Qualidade RAGAS — LAB 3\n"
    "(Barra: média ± desvio | Linha tracejada: meta mínima)",
    fontsize=12, fontweight="bold",
)
ax.grid(axis="y", alpha=0.4, linestyle=":")

# Legenda
patch_ok    = mpatches.Patch(color="#2ecc71", label="Meta atingida")
patch_fail  = mpatches.Patch(color="#e74c3c", label="Abaixo da meta")
patch_meta  = mpatches.Patch(linestyle="--", edgecolor="red",
                              facecolor="none", label="Meta mínima")
ax.legend(handles=[patch_ok, patch_fail, patch_meta], fontsize=10)

plt.tight_layout()
plt.savefig("dashboard_qualidade_lab3.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Gráfico salvo: 'dashboard_qualidade_lab3.png'")

---
## Célula 9 — Padrão de Callback/Decorator com `@observe` do LangFuse

Demonstra como usar o decorator `@observe` do LangFuse combinado com RAGAS para automatizar a monitoração em produção, sem precisar criar traces manualmente.

In [ ]:
# Demonstração do padrão @observe para uso em produção
# O decorator @observe captura automaticamente:
#   - input/output da função
#   - tempo de execução
#   - exceções
# Combinar com RAGAS permite monitoração contínua sem boilerplate.

# Importações necessárias para o padrão de produção
try:
    from langfuse.decorators import observe, langfuse_context
    OBSERVE_DISPONIVEL = True
    print("[OK] langfuse.decorators importado com sucesso.")
except ImportError:
    OBSERVE_DISPONIVEL = False
    print("[AVISO] langfuse.decorators não disponível nesta versão do LangFuse.")
    print("  O padrão abaixo é ilustrativo — use com LangFuse >= 2.x.")


# ── Padrão de callback/decorator para produção ────────────────────────────
# Este código demonstra como integrar em um sistema de produção

if OBSERVE_DISPONIVEL:

    @observe(name="rag-juridico-com-ragas")  # nomeia o trace automaticamente
    def pipeline_producao(
        question: str,
        ground_truth: str,
        os_client,
        llm,
        embedder,
    ) -> dict:
        """
        Função de produção com trace automático via @observe.
        RAGAS calcula métricas e LangFuse as registra no trace atual.
        """
        # Adiciona metadados ao trace via langfuse_context
        langfuse_context.update_current_observation(
            metadata={
                "dominio": "juridico",
                "versao":  "1.0.0",
                "lab":     "LAB3",
            },
            tags=["producao", "rag", "ragas-integrado"],
        )

        # Retrieval
        contextos = pipeline._recuperar_contextos(question)

        # Geração
        answer = pipeline._gerar_resposta(question, contextos)

        # Avaliação RAGAS
        scores = pipeline._avaliar_ragas(question, answer, contextos, ground_truth)

        # Envia scores ao trace atual via langfuse_context
        trace_id = langfuse_context.get_current_trace_id()
        if lf_client and trace_id:
            for nome_metrica, valor in scores.items():
                lf_client.score(
                    trace_id=trace_id,
                    name=nome_metrica,
                    value=valor,
                    comment=f"@observe automático | Meta: {METAS.get(nome_metrica, 0)}",
                )

        return {"answer": answer, "scores": scores, "trace_id": trace_id}

    print("\n[OK] Função pipeline_producao() definida com @observe.")
    print("  Uso: pipeline_producao(question, ground_truth, os_client, llm, embedder)")
    print("  O trace é criado e fechado automaticamente pelo decorator.")

else:
    # Versão alternativa: callback manual via contextmanager
    from contextlib import contextmanager

    @contextmanager
    def trace_ragas_contextmanager(question: str, ground_truth: str):
        """
        Alternativa quando @observe não está disponível.
        Usa context manager para garantir flush mesmo em erros.
        """
        trace = None
        if lf_client:
            trace = lf_client.trace(
                name="rag-juridico-com-ragas",
                input={"question": question},
                tags=["producao", "rag", "ragas-integrado"],
            )
        try:
            yield trace
        finally:
            if lf_client:
                lf_client.flush()  # garante envio mesmo em exceção

    def pipeline_producao_alternativo(question: str, ground_truth: str) -> dict:
        """Versão alternativa usando context manager."""
        with trace_ragas_contextmanager(question, ground_truth) as trace:
            contextos = pipeline._recuperar_contextos(question)
            answer    = pipeline._gerar_resposta(question, contextos)
            scores    = pipeline._avaliar_ragas(question, answer, contextos, ground_truth)

            if trace and lf_client:
                for nome_metrica, valor in scores.items():
                    lf_client.score(
                        trace_id=trace.id,
                        name=nome_metrica,
                        value=valor,
                    )
                trace.update(output={"answer": answer, "scores": scores})

            return {
                "answer":   answer,
                "scores":   scores,
                "trace_id": trace.id if trace else None,
            }

    print("\n[OK] Alternativa via context manager definida.")


# ── Demonstração de execução com o padrão ─────────────────────────────────
print("\nDemonstrando execução com o padrão de callback...")
par_demo = dataset_pares[0]

if OBSERVE_DISPONIVEL:
    resultado_observe = pipeline_producao(
        question=par_demo["question"],
        ground_truth=par_demo["ground_truth"],
        os_client=os_client,
        llm=llm,
        embedder=embedder,
    )
    print(f"  Trace ID (@observe): {resultado_observe.get('trace_id')}")
    print(f"  Scores: {resultado_observe.get('scores')}")
else:
    resultado_cm = pipeline_producao_alternativo(
        question=par_demo["question"],
        ground_truth=par_demo["ground_truth"],
    )
    print(f"  Trace ID (context manager): {resultado_cm.get('trace_id')}")
    print(f"  Scores: {resultado_cm.get('scores')}")

---
## Célula 10 — Exportação dos Resultados em CSV

In [ ]:
import os as _os

CAMINHO_CSV = "ragas_langfuse_integrado_resultados.csv"

# Monta o DataFrame de resultados
linhas = []
for i, resultado in enumerate(resultados_batch, start=1):
    linha = {
        "par":           i,
        "question":      resultado["question"],
        "answer":        resultado["answer"][:200],   # trunca para legibilidade
        "ground_truth":  resultado["ground_truth"][:200],
        "n_contextos":   len(resultado["contexts"]),
        "trace_id":      resultado.get("trace_id", ""),
        "tempo_s":       resultado.get("tempo_s", 0),
        # Métricas RAGAS
        "faithfulness":       resultado["scores"].get("faithfulness",      float("nan")),
        "answer_relevancy":   resultado["scores"].get("answer_relevancy",   float("nan")),
        "context_recall":     resultado["scores"].get("context_recall",     float("nan")),
        "context_precision":  resultado["scores"].get("context_precision",  float("nan")),
        # Status por métrica
        "faithfulness_ok":      resultado["scores"].get("faithfulness",      0.0) >= METAS["faithfulness"],
        "answer_relevancy_ok":  resultado["scores"].get("answer_relevancy",  0.0) >= METAS["answer_relevancy"],
        "context_recall_ok":    resultado["scores"].get("context_recall",    0.0) >= METAS["context_recall"],
        "context_precision_ok": resultado["scores"].get("context_precision", 0.0) >= METAS["context_precision"],
        # Link LangFuse
        "langfuse_url":  (
            f"{LANGFUSE_HOST}/traces/{resultado.get('trace_id')}"
            if resultado.get("trace_id") else ""
        ),
    }
    linhas.append(linha)

df_export = pd.DataFrame(linhas)
df_export.to_csv(CAMINHO_CSV, index=False, encoding="utf-8")

tamanho_kb = _os.path.getsize(CAMINHO_CSV) / 1024
print(f"[OK] Resultados exportados: '{CAMINHO_CSV}' — {tamanho_kb:.1f} KB")
print(f"  Linhas    : {len(df_export)}")
print(f"  Colunas   : {list(df_export.columns)}")

# Prévia das primeiras 3 linhas
print("\nPrévia dos primeiros 3 resultados:")
colunas_preview = ["par", "faithfulness", "answer_relevancy",
                   "context_recall", "context_precision", "trace_id"]
print(df_export[colunas_preview].head(3).to_string(index=False))

# Sumário de metas atingidas
print("\nSumário de metas atingidas por par:")
cols_ok = [c for c in df_export.columns if c.endswith("_ok")]
df_export["todas_metas_ok"] = df_export[cols_ok].all(axis=1)
n_todos_ok = df_export["todas_metas_ok"].sum()
print(f"  Pares com TODAS as metas: {n_todos_ok}/{len(df_export)}")
for col in cols_ok:
    n_ok = df_export[col].sum()
    metrica = col.replace("_ok", "")
    print(f"  {metrica:<22}: {n_ok}/{len(df_export)} pares OK")

---

## Checklist de Conclusão — LAB 3

Revise cada item antes de prosseguir para o **LAB 4 (DeepEval — Testes Unitários)**:

| # | Item | Status |
|---|------|--------|
| 1 | Dependências instaladas sem erros críticos (ragas, langfuse, langchain, opensearch-py) | ☐ |
| 2 | Health checks executados: vLLM, OpenSearch, LangFuse e Embedder com resultado documentado | ☐ |
| 3 | Classe `RAGPipelineComAvaliacao` instanciada com os parâmetros corretos | ☐ |
| 4 | Método `run()` executa: retrieval kNN → geração vLLM → RAGAS 4 métricas → LangFuse Scores API | ☐ |
| 5 | Batch de 10 pares processado com `trace_id` exibido para cada par | ☐ |
| 6 | Scores confirmados visíveis na UI LangFuse (`{LANGFUSE_HOST}/traces/{trace_id}`) | ☐ |
| 7 | Dashboard de qualidade gerado: gráfico de barras com médias vs metas (linha vermelha tracejada) | ☐ |
| 8 | Padrão `@observe` ou context manager demonstrado para uso em produção | ☐ |
| 9 | CSV `ragas_langfuse_integrado_resultados.csv` exportado com scores e links LangFuse | ☐ |

> **Nota ABNT:** Este laboratório segue as diretrizes da ABNT NBR 6023:2018 e ABNT NBR 14724:2011. Os resultados aqui obtidos constituem a **integração de observabilidade** do projeto de avaliação contínua de pipelines RAG aplicados ao domínio jurídico e de segurança pública, servindo como base para o monitoramento em ambiente de produção.